In [1]:
import os
import xml.etree.ElementTree as ET
from glob import glob

# 路径配置（根据你的数据集修改）
voc_path = '/kaggle/input/xiaoguanjiefenji/VOC2007' # 你的数据路径
output_path = '/kaggle/working/yolo_dataset'

# classes = ['joint'] # 替换为你 XML 里的 label 名
classes = ['Radius', 'Ulna', 'MCPFirst', 'ProximalPhalanx', 'MCP', 'DistalPhalanx', 'MiddlePhalanx']

def convert_voc_to_yolo():
    os.makedirs(f'{output_path}/labels/train', exist_ok=True)
    os.makedirs(f'{output_path}/images/train', exist_ok=True)
    os.makedirs(f'{output_path}/labels/test', exist_ok=True)
    os.makedirs(f'{output_path}/images/test', exist_ok=True)
    os.makedirs(f'{output_path}/labels/val', exist_ok=True)
    os.makedirs(f'{output_path}/images/val', exist_ok=True)
    
    xml_files = glob(f'{voc_path}/Annotations/*.xml')
    import random
    # xml_files.shuffle()
    # classes = []
    random.seed(20)
    random.shuffle(xml_files)
    test_rate = 0.15
    val_rate  = 0.15
    end_idx1 = int(len(xml_files)*(1 - test_rate - val_rate))
    end_idx2 = int(len(xml_files)*(1 - val_rate))
    train_files = xml_files[:end_idx1]
    
    val_files = xml_files[end_idx1:end_idx2]
    test_files = xml_files[end_idx2:]
    
    for xml_file in train_files:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        img_id = root.find('filename').text.split('.')[0]
        
        # 处理图片和标签
        with open(f'{output_path}/labels/train/{img_id}.txt', 'w+') as f:
            size = root.find('size')
            w = int(size.find('width').text)
            h = int(size.find('height').text)
            
            for obj in root.iter('object'):
                cls = obj.find('name').text
                if cls not in classes: classes.append(cls)
                cls_id = classes.index(cls)
                xmlbox = obj.find('bndbox')
                b = (float(xmlbox.find('xmin').text), float(xmlbox.find('xmax').text), 
                     float(xmlbox.find('ymin').text), float(xmlbox.find('ymax').text))
                # 转换为 YOLO 归一化坐标
                bb = ((b[0]+b[1])/2/w, (b[2]+b[3])/2/h, (b[1]-b[0])/w, (b[3]-b[2])/h)
                f.write(f"{cls_id} {' '.join([f'{a:.6f}' for a in bb])}\n")
        
        # 建立图片软链接到工作区
        img_src = f"{voc_path}/JPEGImages/{img_id}.png" # 注意后缀是.jpg还是.png
        if os.path.exists(img_src) and not os.path.lexists(os.path.join(img_src, f"{output_path}/images/train/{img_id}.png")):
            os.symlink(img_src, f"{output_path}/images/train/{img_id}.png" )
    for xml_file in test_files:
        # print(xml_file)
        tree = ET.parse(xml_file)
        root = tree.getroot()
        img_id = root.find('filename').text.split('.')[0]
    
            
            # 处理图片和标签
        with open(f'{output_path}/labels/test/{img_id}.txt', 'w+') as f:
            size = root.find('size')
            w = int(size.find('width').text)
            h = int(size.find('height').text)
            # print(size)
            
            for obj in root.iter('object'):
                cls = obj.find('name').text
                if cls not in classes: classes.append(cls)
                cls_id = classes.index(cls)
                xmlbox = obj.find('bndbox')
                b = (float(xmlbox.find('xmin').text), float(xmlbox.find('xmax').text), 
                     float(xmlbox.find('ymin').text), float(xmlbox.find('ymax').text))
                # 转换为 YOLO 归一化坐标
                bb = ((b[0]+b[1])/2/w, (b[2]+b[3])/2/h, (b[1]-b[0])/w, (b[3]-b[2])/h)
                # print(f"{cls_id} {' '.join([f'{a:.6f}' for a in bb])}\n")
                # break
                # print('T')
                f.write(f"{cls_id} {' '.join([f'{a:.6f}' for a in bb])}\n")
                # 建立图片软链接到工作区
            img_src = f"{voc_path}/JPEGImages/{img_id}.png" # 注意后缀是.jpg还是.png
        if os.path.exists(img_src) and not os.path.lexists(os.path.join(img_src, f"{output_path}/images/test/{img_id}.png")):
            os.symlink(img_src, f"{output_path}/images/test/{img_id}.png")
    for xml_file in val_files:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        img_id = root.find('filename').text.split('.')[0]
    
            
            # 处理图片和标签
        with open(f'{output_path}/labels/val/{img_id}.txt', 'w+') as f:
            size = root.find('size')
            w = int(size.find('width').text)
            h = int(size.find('height').text)
            
            for obj in root.iter('object'):
                cls = obj.find('name').text
                if cls not in classes: classes.append
                cls_id = classes.index(cls)
                xmlbox = obj.find('bndbox')
                b = (float(xmlbox.find('xmin').text), float(xmlbox.find('xmax').text), 
                     float(xmlbox.find('ymin').text), float(xmlbox.find('ymax').text))
                # 转换为 YOLO 归一化坐标
                bb = ((b[0]+b[1])/2/w, (b[2]+b[3])/2/h, (b[1]-b[0])/w, (b[3]-b[2])/h)
                f.write(f"{cls_id} {' '.join([f'{a:.6f}' for a in bb])}\n")
                # 建立图片软链接到工作区
            img_src = f"{voc_path}/JPEGImages/{img_id}.png" # 注意后缀是.jpg还是.png
        if os.path.exists(img_src) and not os.path.lexists(os.path.join(img_src, f"{output_path}/images/val/{img_id}.png")):
            os.symlink(img_src, f"{output_path}/images/val/{img_id}.png")
    return classes
classes = convert_voc_to_yolo()
print(classes)
print("数据转换完成！")

['Radius', 'Ulna', 'MCPFirst', 'ProximalPhalanx', 'MCP', 'DistalPhalanx', 'MiddlePhalanx']
数据转换完成！


In [2]:
# import pandas as pd 
# file = pd.read_csv("/kaggle/working/hand.yaml")
# print(file.head())

In [3]:
import pandas as pd

# 修正后的代码
try:
    # sep=' ' 表示以空格分隔
    # names 给列起个名字，方便查看：类ID, 中心X, 中心Y, 宽度, 高度
    f = open("/kaggle/working/yolo_dataset/labels/test/14716.txt",'r')
    print(f.read())
except FileNotFoundError:
    print("找不到该文件，请检查路径是否正确。")

1 0.368874 0.832757 0.125537 0.133795
0 0.503869 0.828143 0.209802 0.137255
2 0.687446 0.666667 0.153912 0.102653
4 0.327601 0.506920 0.104901 0.098039
4 0.426053 0.482122 0.098882 0.098039
4 0.523646 0.453576 0.101462 0.106690
4 0.634136 0.455017 0.109200 0.099193
3 0.808255 0.515571 0.123818 0.077278
3 0.652623 0.371396 0.123818 0.087659
3 0.531814 0.370242 0.117799 0.089965
3 0.423044 0.403403 0.111780 0.087082
3 0.298796 0.436563 0.117799 0.077278
6 0.238607 0.337082 0.098882 0.061707
6 0.404127 0.273068 0.111780 0.070934
6 0.543852 0.233564 0.117799 0.080738
6 0.679708 0.249423 0.121238 0.079008
5 0.866294 0.386967 0.145314 0.151096
5 0.684867 0.144752 0.131556 0.113033
5 0.535684 0.106690 0.137575 0.136101
5 0.408426 0.155421 0.128977 0.126298
5 0.221840 0.239908 0.122098 0.118800



In [4]:
# 按照你提供的顺序重新生成 YAML
classes = ['Radius', 'Ulna', 'MCPFirst', 'ProximalPhalanx', 'MCP', 'DistalPhalanx', 'MiddlePhalanx']

yaml_content = f"""
path: /kaggle/working/yolo_dataset
train: images/train
val: images/val
test: images/test
names:
{chr(10).join([f'  {i}: {name}' for i, name in enumerate(classes)])}
"""

with open('/kaggle/working/hand.yaml', 'w') as f:
    f.write(yaml_content.strip())

print("✅ hand.yaml 已更新，当前类别列表：")
for i, name in enumerate(classes):
    print(f"  索引 {i}: {name}")

# ！！！一定要删缓存，否则 YOLO 还会用旧的单类索引去匹配
!rm -rf /kaggle/working/yolo_dataset/labels/*.cache

✅ hand.yaml 已更新，当前类别列表：
  索引 0: Radius
  索引 1: Ulna
  索引 2: MCPFirst
  索引 3: ProximalPhalanx
  索引 4: MCP
  索引 5: DistalPhalanx
  索引 6: MiddlePhalanx


In [5]:
!pip install ultralytics -q  # -q 是安静模式，不会刷出一堆安装日志

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.1 MB/s eta 0:00:00


In [6]:

# from ultralytics import YOLO



# # 1. 加载预训练模型

# model = YOLO('yolov8s.pt') 


# results = model.train(
#     data='/kaggle/working/hand.yaml', 
#     epochs=100,           # 增加到100轮，因为增强多了模型学得慢，但上限更高
#     imgsz=1024, 
#     batch=8,
#     device=0,
#     patience=20,
    
#     # --- 空间增强 ---
#     degrees=20.0,         # 旋转角度范围 (+/- 20°)
#     shear=10.0,           # 仿射变换
#     perspective=0.0005,    # 微小的透视变换
#     flipud=0.5,           # 增加上下翻转，关节在画面里方向多变
#     fliplr=0.5,           # 左右翻转（默认开启，这里显式写出）
    
#     # --- 颜色与噪点增强 ---
#     hsv_h=0.015,          # 色调变化 (0.0 - 1.0)
#     hsv_s=0.7,            # 饱和度变化
#     hsv_v=0.4,            # 亮度变化
    
#     # --- 混合增强 (强力推荐) ---
#     mosaic=1.0,           # 默认1.0，四合一拼接，对小目标极好
#     mixup=0.1,            # 图像融合，防止过拟合
    
#     # --- 训练策略优化 ---
#     warmup_epochs=3.0,     # 前3轮热身，防止梯度爆炸
#     close_mosaic=10,      # 最后10轮关闭马赛克，让模型回归真实分布进行微调
# )

In [7]:
# import matplotlib.pyplot as plt
# import cv2
# import os

# # 1. 设置路径
# img_dir = '/kaggle/working/yolo_dataset/images/train'
# lab_dir = '/kaggle/working/yolo_dataset/labels/train'

# # 2. 获取第一个文件名（确保目录不为空）
# img_files = sorted(os.listdir(img_dir))
# if not img_files:
#     print("❌ 文件夹是空的，请检查路径！")
# else:
#     img_name = img_files[0]
#     img_path = os.path.join(img_dir, img_name)
#     # 找到对应的标签文件 (.jpg -> .txt)
#     lab_path = os.path.join(lab_dir, os.path.splitext(img_name)[0] + '.txt')

#     # 3. 使用 OpenCV 读取图片
#     img = cv2.imread(img_path)
#     img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#     h, w, _ = img.shape

#     # 4. 如果有标签，读取并画框
#     if os.path.exists(lab_path):
#         with open(lab_path, 'r') as f:
#             for line in f.readlines():
#                 cls, x, y, nw, nh = map(float, line.split())
                
#                 # YOLO 归一化坐标转像素坐标
#                 x1 = int((x - nw/2) * w)
#                 y1 = int((y - nh/2) * h)
#                 x2 = int((x + nw/2) * w)
#                 y2 = int((y + nh/2) * h)
                
#                 # 画矩形框 (绿色，粗细为 2)
#                 cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
#                 cv2.putText(img, "joint", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
#     else:
#         print(f"⚠️ 未找到对应的标签文件: {lab_path}")

#     # 5. 展示图片
#     plt.figure(figsize=(10, 10))
#     plt.imshow(img)
#     plt.title(f"Image: {img_name}")
#     plt.axis('off')
#     plt.show()

In [8]:
# 清理所有缓存文件
!rm -rf /kaggle/working/yolo_dataset/labels/*.cache
!rm -rf /kaggle/working/yolo_dataset/images/*.cache
print("✅ 缓存已清理")

✅ 缓存已清理


In [9]:
import os
from glob import glob

# 1. 彻底删除所有缓存
!find /kaggle/working/yolo_dataset -name "*.cache" -delete

# 2. 定义路径
base_path = '/kaggle/working/yolo_dataset'
train_img_dir = os.path.join(base_path, 'images/train')
train_lab_dir = os.path.join(base_path, 'labels/train')

# 3. 统计并列出前 5 个匹配项
imgs = sorted(glob(os.path.join(train_img_dir, '*')))
# 过滤掉非图片文件（如 .ipynb_checkpoints）
imgs = [f for f in imgs if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]

print(f"📊 扫描报告:")
print(f"有效图片数量: {len(imgs)}")

valid_pairs = 0
for img_p in imgs[:10]: # 检查前10个
    img_name = os.path.basename(img_p)
    name_wo_ext = os.path.splitext(img_name)[0]
    lab_p = os.path.join(train_lab_dir, name_wo_ext + '.txt')
    
    if os.path.exists(lab_p):
        valid_pairs += 1
    else:
        print(f"❌ 缺失标签: {img_name} -> 找不到 {name_wo_ext}.txt")

print(f"✅ 前10张图中匹配成功的对数: {valid_pairs}")

📊 扫描报告:
有效图片数量: 616
✅ 前10张图中匹配成功的对数: 10


In [10]:
import os

def sanitize_yolo_labels(label_path):
    print(f"正在清洗标签路径: {label_path}")
    count = 0
    for file in os.listdir(label_path):
        if file.endswith('.txt'):
            file_path = os.path.join(label_path, file)
            with open(file_path, 'r') as f:
                lines = f.readlines()
            
            # 过滤掉空行，去掉每行前后的空格，确保只有数字和空格
            clean_lines = [l.strip() for l in lines if l.strip()]
            
            with open(file_path, 'w', newline='\n') as f:
                f.write('\n'.join(clean_lines) + '\n')
            count += 1
    print(f"✅ 清洗完成，共处理 {count} 个文件")

# 执行清洗
sanitize_yolo_labels('/kaggle/working/yolo_dataset/labels/train')
sanitize_yolo_labels('/kaggle/working/yolo_dataset/labels/test') # 假设 val 也指向这

# 再次强力删除缓存（非常重要）
!rm -rf /kaggle/working/yolo_dataset/labels/*.cache
!rm -rf /kaggle/working/yolo_dataset/images/*.cache
!rm -rf /kaggle/working/yolo_dataset/*.cache

正在清洗标签路径: /kaggle/working/yolo_dataset/labels/train
✅ 清洗完成，共处理 616 个文件
正在清洗标签路径: /kaggle/working/yolo_dataset/labels/test
✅ 清洗完成，共处理 133 个文件


In [11]:
import os

label_dir = '/kaggle/working/yolo_dataset/labels/train'
files = [f for f in os.listdir(label_dir) if f.endswith('.txt')][:5]

print("📊 标签文件内部抽查:")
for f in files:
    with open(os.path.join(label_dir, f), 'r') as file:
        content = file.readline().strip()
        print(f"文件名: {f} | 内容: {content}")

📊 标签文件内部抽查:
文件名: 1942.txt | 内容: 0 0.441215 0.836840 0.183620 0.133562
文件名: 6936.txt | 内容: 6 0.595238 0.313774 0.098214 0.077686
文件名: 1939.txt | 内容: 6 0.662153 0.221869 0.102378 0.081703
文件名: 2031.txt | 内容: 4 0.615032 0.471396 0.078827 0.102212
文件名: 6954.txt | 内容: 0 0.499112 0.850284 0.207815 0.194567


In [12]:
from ultralytics import YOLO
import pandas as pd

# 路径配置
yaml_path = '/kaggle/working/hand.yaml'
lr_candidates = [0.01, 0.005, 0.001]
results_summary = []

for lr in lr_candidates:
    print(f"\n" + "="*30)
    print(f"🚀 实验开始: lr0 = {lr}")
    print("="*30)
    

    from ultralytics import YOLO
    
    
    
    # 1. 加载预训练模型
    
    model = YOLO('yolov8s.pt') 
    
    
    results = model.train(
        data='/kaggle/working/hand.yaml', 
        lr0=lr,
        project='lr_comparison',
        name=f'train_lr_{lr}',
        epochs=100,           # 增加到100轮，因为增强多了模型学得慢，但上限更高
        imgsz=1024, 
        batch=8,
        device=0,
        patience=20,
        
        # --- 空间增强 ---
        degrees=20.0,         # 旋转角度范围 (+/- 20°)
        shear=10.0,           # 仿射变换
        perspective=0.0005,    # 微小的透视变换
        flipud=0.5,           # 增加上下翻转，关节在画面里方向多变
        fliplr=0.5,           # 左右翻转（默认开启，这里显式写出）
        
        # --- 颜色与噪点增强 ---
        hsv_h=0.015,          # 色调变化 (0.0 - 1.0)
        hsv_s=0.7,            # 饱和度变化
        hsv_v=0.4,            # 亮度变化
        
        # --- 混合增强 (强力推荐) ---
        mosaic=1.0,           # 默认1.0，四合一拼接，对小目标极好
        mixup=0.1,            # 图像融合，防止过拟合
        
            # --- 训练策略优化 ---
        warmup_epochs=3.0,     # 前3轮热身，防止梯度爆炸
        close_mosaic=10,      # 最后10轮关闭马赛克，让模型回归真实分布进行微调
    )
        
    # # 2. 验证：加载刚训练好的 best.pt 并在测试集上跑一遍
    # best_model_path = f'/kaggle/working/lr_comparison/train_lr_{lr}/weights/best.pt'
    # best_model = YOLO(best_model_path)
    
    # # 执行测试集验证
    # metrics = best_model.val(data=yaml_path, split='test', imgsz=1024, plots=False)
    
    # # 3. 记录核心数据
    # results_summary.append({
    #     'learning_rate': lr,
    #     'mAP50': metrics.box.map50,
    #     'mAP50-95': metrics.box.map,
    #     'Precision': metrics.box.mp,
    #     'Recall': metrics.box.mr
    # })
# --- 修改后的验证逻辑 ---
    
    # 1. 直接从 results 对象获取本次训练真实的保存路径
    # results.save_dir 会自动指向类似 /kaggle/working/runs/detect/lr_comparison/train_lr_0.019 的位置
    best_model_path = os.path.join(results.save_dir, 'weights', 'best.pt')
    
    print(f"🔎 正在尝试加载模型: {best_model_path}")
    
    if os.path.exists(best_model_path):
        # 2. 加载模型
        best_model = YOLO(best_model_path)
        
        # 3. 执行测试集验证 (注意 split='test')
        metrics = best_model.val(data=yaml_path, split='test', imgsz=1024, plots=False)
        
        # 4. 记录核心数据
        results_summary.append({
            'learning_rate': lr,
            'mAP50': metrics.box.map50,
            'mAP50-95': metrics.box.map,
            'Precision': metrics.box.mp,
            'Recall': metrics.box.mr
        })
        print(f"✅ 学习率 {lr} 验证完成，mAP50: {metrics.box.map50:.4f}")
    else:
        print(f"❌ 错误：找不到权重文件 {best_model_path}，请检查训练日志。")

# 4. 汇总展示
df = pd.DataFrame(results_summary)
print("\n" + "🏆 最终实验对比结果 " + "🏆")
print(df.sort_values(by='mAP50', ascending=False)) # 按 mAP50 从高到低排序

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

🚀 实验开始: lr0 = 0.01
Ultralytics 8.4.32 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/hand.yaml, degrees=20.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7,